In [0]:
# Read credentials from Key Vault via secret scope
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

print("Secrets loaded successfully")

Secrets loaded successfully


In [0]:
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

RAW_PATH       = f"abfss://raw@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"
CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"

print("✅ ADLS configured successfully")
print(f"RAW       → {RAW_PATH}")
print(f"PROCESSED → {PROCESSED_PATH}")
print(f"CURATED   → {CURATED_PATH}")

✅ ADLS configured successfully
RAW       → abfss://raw@azurelabadls225.dfs.core.windows.net
PROCESSED → abfss://processed@azurelabadls225.dfs.core.windows.net
CURATED   → abfss://curated@azurelabadls225.dfs.core.windows.net


In [0]:
dbutils.fs.ls(f"{RAW_PATH}/")
print("✅ Connected to ADLS successfully!")

✅ Connected to ADLS successfully!


In [0]:
import urllib.request
import os

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data"
months = [f"{m:02d}" for m in range(1, 13)]

for month in months:
    filename = f"yellow_tripdata_2023-{month}.parquet"
    url = f"{base_url}/{filename}"
    local_path = f"/tmp/{filename}"
    adls_path = f"{RAW_PATH}/yellow_taxi/{filename}"
    
    # Check if already exists
    try:
        dbutils.fs.ls(adls_path)
        print(f"⏭️  Already exists: {filename}")
        continue
    except:
        pass
    
    # Download to driver temp storage
    print(f"⬇️  Downloading {filename}...")
    urllib.request.urlretrieve(url, local_path)
    
    # Copy to ADLS
    dbutils.fs.cp(f"file://{local_path}", adls_path)
    os.remove(local_path)
    print(f"✅ Uploaded: {filename}")

print("\n🎉 All files loaded to ADLS!")

⬇️  Downloading yellow_tripdata_2023-01.parquet...
✅ Uploaded: yellow_tripdata_2023-01.parquet
⬇️  Downloading yellow_tripdata_2023-02.parquet...
✅ Uploaded: yellow_tripdata_2023-02.parquet
⬇️  Downloading yellow_tripdata_2023-03.parquet...
✅ Uploaded: yellow_tripdata_2023-03.parquet
⬇️  Downloading yellow_tripdata_2023-04.parquet...
✅ Uploaded: yellow_tripdata_2023-04.parquet
⬇️  Downloading yellow_tripdata_2023-05.parquet...
✅ Uploaded: yellow_tripdata_2023-05.parquet
⬇️  Downloading yellow_tripdata_2023-06.parquet...
✅ Uploaded: yellow_tripdata_2023-06.parquet
⬇️  Downloading yellow_tripdata_2023-07.parquet...
✅ Uploaded: yellow_tripdata_2023-07.parquet
⬇️  Downloading yellow_tripdata_2023-08.parquet...
✅ Uploaded: yellow_tripdata_2023-08.parquet
⬇️  Downloading yellow_tripdata_2023-09.parquet...
✅ Uploaded: yellow_tripdata_2023-09.parquet
⬇️  Downloading yellow_tripdata_2023-10.parquet...
✅ Uploaded: yellow_tripdata_2023-10.parquet
⬇️  Downloading yellow_tripdata_2023-11.parquet...

In [0]:
import urllib.request

url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
local_path = "/tmp/taxi_zone_lookup.csv"
adls_path = f"{RAW_PATH}/lookup/taxi_zone_lookup.csv"

urllib.request.urlretrieve(url, local_path)
dbutils.fs.cp(f"file://{local_path}", adls_path)

print("✅ Zone lookup uploaded!")

# Preview it
df = spark.read.option("header", True).csv(adls_path)
df.show(5)

✅ Zone lookup uploaded!
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows

